# Classification d'images de la langue des signes américaine (ASL) à l'aide d'un CNN

*American Sign Language Image Classification Using Convolutional Neural Networks*

Master ISI — Module « Intelligence Artificielle et Innovation »

Dataset : [Sign Language MNIST (Kaggle)](https://www.kaggle.com/datasets/datamunge/sign-language-mnist)


## 1. Introduction

**Responsable : Commun**

### Contexte

La langue des signes américaine (ASL) utilise notamment un **alphabet manuel** : chaque lettre correspond à une configuration de la main. Reconnaître automatiquement ces configurations à partir d'images est un problème classique de **vision par ordinateur**.

### Problématique

*Un réseau de neurones convolutif (CNN) simple peut-il reconnaître correctement, à partir d'une image 28 × 28 en niveaux de gris, la lettre statique de l'ASL représentée par une main ?*

### Objectifs

1. Charger, vérifier et explorer le dataset **Sign Language MNIST**.
2. Prétraiter les images et contrôler les risques de **fuite de données**.
3. Construire un **CNN** simple et justifier son architecture.
4. Entraîner le modèle en augmentant progressivement le volume de données.
5. L'évaluer **une seule fois** sur un jeu de test indépendant (accuracy, précision, rappel, F1, matrice de confusion).
6. Analyser ses erreurs et l'utiliser sur une nouvelle image.

### Périmètre

- **24 classes** : les lettres de A à Y, sauf **J et Z**, qui demandent un mouvement et ne peuvent pas être représentées par une image fixe.
- Le projet classe des **images individuelles**. Il ne traduit pas des mots ou des phrases en langue des signes.
- Les résultats obtenus sur ce dataset ne prouvent pas une reconnaissance de l'ASL en conditions réelles (voir les limites en conclusion).

### Outils

Python, TensorFlow/Keras, Google Colab (GPU T4).


## 2. Importation des bibliothèques et configuration

**Responsable : Étudiante 1**

Cette section prépare l'environnement de travail :

- **NumPy** : calcul sur les tableaux (les images sont des tableaux de pixels).
- **Pandas** : lecture des fichiers CSV du dataset.
- **Matplotlib** : affichage des images et des graphiques.
- **scikit-learn** : séparation train/validation et métriques d'évaluation.
- **TensorFlow / Keras** : construction et entraînement du CNN.

On vérifie aussi les **versions** des bibliothèques (pour la reproductibilité) et la **présence d'un GPU**, qui accélère fortement l'entraînement d'un CNN.

Enfin, on fixe une **graine aléatoire** (`SEED`) : les tirages aléatoires (mélange des données, initialisation des poids) deviennent reproductibles d'une exécution à l'autre. Sur GPU, une petite variation peut subsister, ce qui est normal.

> Avant d'exécuter : **Exécution → Modifier le type d'exécution → GPU**.


In [ ]:
# --- Bibliothèques ---
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import tensorflow as tf
from tensorflow import keras

# --- Versions (pour la reproductibilité) ---
print("NumPy        :", np.__version__)
print("Pandas       :", pd.__version__)
print("scikit-learn :", sklearn.__version__)
print("TensorFlow   :", tf.__version__)
print("Keras        :", keras.__version__)

# --- Vérification du GPU ---
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print("GPU disponible :", gpus)
else:
    print("Aucun GPU détecté : activer le GPU dans Exécution > Modifier le type d'exécution.")

# --- Graine aléatoire ---
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("Graine fixée :", SEED)


**Résultat obtenu (exécution sur Google Colab, GPU T4) :**

| Bibliothèque | Version |
|---|---|
| NumPy | 2.4.6 |
| Pandas | 2.2.3 |
| scikit-learn | 1.6.1 |
| TensorFlow | 2.21.0 |
| Keras | 3.13.2 |

Un GPU est détecté (`/physical_device:GPU:0`) et la graine est fixée à 42. L'environnement est prêt pour l'entraînement du CNN.


## 3. Chargement des données

**Responsable : Étudiante 1**

Le dataset est téléchargé depuis Kaggle avec **KaggleHub**, la bibliothèque officielle de Kaggle. Elle place les fichiers dans un dossier local de Colab.

Ensuite, on :

1. liste les fichiers téléchargés pour **identifier les CSV** réellement présents ;
2. lit `sign_mnist_train.csv` et `sign_mnist_test.csv` avec **Pandas** ;
3. vérifie les **dimensions** (nombre de lignes et de colonnes) ;
4. vérifie les **colonnes** (une colonne `label` + 784 colonnes de pixels attendues) ;
5. cherche les **valeurs manquantes** ;
6. affiche les **labels réellement présents** et la **plage des valeurs des pixels**.

Les chiffres annoncés par la fiche Kaggle (27 455 images d'entraînement, 7 172 de test, 24 classes) ne sont considérés comme confirmés qu'après cette vérification.


In [ ]:
import kagglehub

# --- Téléchargement du dataset ---
dataset_path = kagglehub.dataset_download("datamunge/sign-language-mnist")
print("Dossier du dataset :", dataset_path)

# --- Liste de tous les fichiers téléchargés ---
csv_files = []
print("\nFichiers trouvés :")
for root, dirs, files in os.walk(dataset_path):
    for f in files:
        full_path = os.path.join(root, f)
        print(" -", os.path.relpath(full_path, dataset_path))
        if f.endswith(".csv"):
            csv_files.append(full_path)

# --- Sélection des deux fichiers CSV ---
def find_csv(name):
    matches = [p for p in csv_files if os.path.basename(p) == name]
    if not matches:
        raise FileNotFoundError(f"{name} introuvable dans {dataset_path}")
    print(f"{name} : {len(matches)} copie(s) trouvée(s), utilisée :", os.path.relpath(matches[0], dataset_path))
    return matches[0]

print()
train_csv = find_csv("sign_mnist_train.csv")
test_csv = find_csv("sign_mnist_test.csv")

# --- Lecture avec Pandas ---
train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)

# --- Dimensions ---
print("\nDimensions train :", train_df.shape)
print("Dimensions test  :", test_df.shape)

# --- Colonnes ---
print("\nPremières colonnes :", train_df.columns[:4].tolist())
print("Dernières colonnes :", train_df.columns[-3:].tolist())
print("Mêmes colonnes dans train et test :", list(train_df.columns) == list(test_df.columns))

# --- Valeurs manquantes ---
print("\nValeurs manquantes train :", int(train_df.isnull().sum().sum()))
print("Valeurs manquantes test  :", int(test_df.isnull().sum().sum()))

# --- Labels réellement présents ---
print("\nLabels train :", sorted(train_df["label"].unique()))
print("Nombre de labels train :", train_df["label"].nunique())
print("Labels test  :", sorted(test_df["label"].unique()))
print("Nombre de labels test  :", test_df["label"].nunique())

# --- Valeurs des pixels ---
pixels_train = train_df.drop(columns="label").values
print("\nPixels train : min =", pixels_train.min(), "| max =", pixels_train.max(), "| type =", pixels_train.dtype)

train_df.head()


**Résultats obtenus :**

- Le dossier téléchargé contient les deux CSV attendus, présents en **deux copies** (à la racine et dans un sous-dossier), ainsi que trois images d'illustration. Les fichiers de la racine sont utilisés.
- **Train : 27 455 lignes × 785 colonnes. Test : 7 172 lignes × 785 colonnes.** Ces chiffres confirment ceux annoncés par la fiche Kaggle (34 627 images au total).
- Les colonnes sont `label` puis `pixel1` à `pixel784` (784 = 28 × 28), identiques dans les deux fichiers.
- **Aucune valeur manquante** dans les deux fichiers.
- **24 labels** dans chaque fichier : les entiers de **0 à 24 sauf 9**. Le label correspond à la position de la lettre dans l'alphabet (0 = A, 1 = B…). Le 9 (J) est absent et le 25 (Z) n'existe pas : cela confirme l'exclusion des deux lettres qui demandent un mouvement.
- Les pixels sont des entiers entre **0 et 255** (niveaux de gris sur 8 bits).

**Conséquence pour la suite :** les labels ne sont pas consécutifs (il manque le 9). Un réseau avec 24 sorties attend des indices de 0 à 23 : il faudra **réencoder les labels** au prétraitement (section 5).


## 4. Exploration des données

**Responsable : Étudiante 1**

Avant de construire le modèle, on observe les données pour vérifier qu'elles correspondent à ce qu'on attend :

1. **Correspondance label → lettre** : on vérifie que chaque label numérique correspond bien à une lettre.
2. **Nombre d'images par classe** : un dataset très déséquilibré pousserait le modèle à favoriser les classes majoritaires.
3. **Exemples d'images** : on reconstruit chaque ligne de 784 pixels en une image 28 × 28 pour voir les signes.
4. **Distribution des valeurs des pixels** : utile pour choisir la normalisation.

Les graphiques sont aussi enregistrés dans le dossier `results/`.


In [ ]:
os.makedirs("results", exist_ok=True)

# --- 1. Correspondance label -> lettre ---
def label_to_letter(label):
    return chr(ord("A") + int(label))

labels_presents = sorted(train_df["label"].unique())
print("Correspondance label -> lettre :")
print({int(l): label_to_letter(l) for l in labels_presents})

# --- 2. Nombre d'images par classe ---
counts = pd.DataFrame({
    "lettre": [label_to_letter(l) for l in labels_presents],
    "train": train_df["label"].value_counts().reindex(labels_presents, fill_value=0).values,
    "test": test_df["label"].value_counts().reindex(labels_presents, fill_value=0).values,
}, index=labels_presents)
counts.index.name = "label"
print("\nNombre d'images par classe :")
print(counts.to_string())

l_min, l_max = counts["train"].idxmin(), counts["train"].idxmax()
print(f"\nTrain : minimum = {counts['train'].min()} ({label_to_letter(l_min)}), "
      f"maximum = {counts['train'].max()} ({label_to_letter(l_max)}), "
      f"rapport max/min = {counts['train'].max() / counts['train'].min():.2f}")
l_min, l_max = counts["test"].idxmin(), counts["test"].idxmax()
print(f"Test  : minimum = {counts['test'].min()} ({label_to_letter(l_min)}), "
      f"maximum = {counts['test'].max()} ({label_to_letter(l_max)}), "
      f"rapport max/min = {counts['test'].max() / counts['test'].min():.2f}")

# Graphique de la distribution des classes
x = np.arange(len(counts))
plt.figure(figsize=(14, 4))
plt.bar(x - 0.2, counts["train"], width=0.4, label="train")
plt.bar(x + 0.2, counts["test"], width=0.4, label="test")
plt.xticks(x, counts["lettre"])
plt.xlabel("Lettre")
plt.ylabel("Nombre d'images")
plt.title("Nombre d'images par classe")
plt.legend()
plt.tight_layout()
plt.savefig("results/distribution_classes.png", dpi=200)
plt.show()

# --- 3. Un exemple d'image par classe ---
fig, axes = plt.subplots(3, 8, figsize=(12, 5))
for ax, l in zip(axes.ravel(), labels_presents):
    ligne = train_df[train_df["label"] == l].iloc[0]
    image = ligne.drop("label").values.reshape(28, 28)
    ax.imshow(image, cmap="gray", interpolation="nearest", vmin=0, vmax=255)
    ax.set_title(f"{label_to_letter(l)} (label {l})", fontsize=9)
    ax.axis("off")
plt.suptitle("Un exemple d'image par classe (train)")
plt.tight_layout()
plt.savefig("results/exemples_par_classe.png", dpi=200)
plt.show()

# --- 4. Distribution des valeurs des pixels ---
plt.figure(figsize=(7, 3.5))
plt.hist(pixels_train.ravel(), bins=64, color="gray")
plt.xlabel("Valeur du pixel (0 = noir, 255 = blanc)")
plt.ylabel("Nombre de pixels")
plt.title("Distribution des valeurs des pixels (train)")
plt.tight_layout()
plt.savefig("results/distribution_pixels.png", dpi=200)
plt.show()
print("Moyenne des pixels :", round(pixels_train.mean(), 1), "| écart-type :", round(pixels_train.std(), 1))


**Résultats obtenus :**

- La correspondance label → lettre est cohérente : 0 = A, …, 8 = I, puis 10 = K, …, 24 = Y.
- **Train** : environ 960 (E) à 1 290 (R) images par lettre (lecture du graphique). Les classes sont **globalement équilibrées** ; l'accuracy est donc une métrique adaptée.
- **Test** : de **144 images (R) à 498 (E)**. Le test est moins équilibré que le train (R est la lettre la plus fréquente du train mais la plus rare du test). On rapportera donc aussi les moyennes **macro** (chaque lettre compte autant) en section 9.
- Les exemples confirment des mains **recadrées**, en niveaux de gris, sur des fonds et éclairages variés.


## 5. Prétraitement

**Responsable : Étudiante 1**

Cette section transforme les tableaux Pandas en données utilisables par le CNN.

1. **Séparer** les labels et les pixels.
2. **Chercher les doublons exacts** : une même image présente deux fois dans le train, ou à la fois dans le train et le test. Une image du test déjà vue à l'entraînement fausserait l'évaluation : c'est une **fuite de données**. On retire ces doublons du train ; le test n'est jamais modifié.
3. **Réencoder les labels** : les labels d'origine vont de 0 à 24 sans le 9. Le réseau a 24 sorties, numérotées de 0 à 23. On crée donc une correspondance label d'origine → indice consécutif, et on garde l'ordre des lettres dans `class_names`.
4. **Remettre en forme les images** : 784 pixels → tableau `28 × 28 × 1`. Le `1` est le **canal** : une image en niveaux de gris n'a qu'un canal (une image couleur en aurait 3).
5. **Normaliser** : diviser par 255 pour avoir des pixels entre 0 et 1. Des valeurs petites et de même échelle rendent l'apprentissage plus stable.
6. **Séparer train / validation** (80 % / 20 %) de manière **stratifiée** : chaque lettre garde la même proportion dans les deux ensembles. La validation sert à régler le modèle ; le **test officiel reste intact** jusqu'à l'évaluation finale.
7. **Construire un sous-ensemble équilibré** de 300 images par classe pour le premier entraînement.
8. **Mesurer la proximité entre validation et entraînement** : le dataset est fabriqué à partir de 1 704 photos, chacune déclinée en de nombreuses variantes. Des variantes d'une même photo peuvent se retrouver à la fois dans le train et la validation. On ne peut pas les retirer (le CSV n'indique pas la photo d'origine), mais on peut mesurer si les images de validation sont **plus proches** du train que les images de test.


In [ ]:
from sklearn.model_selection import train_test_split

# --- 5.1 Séparer labels et pixels ---
y_train_raw = train_df["label"].values
X_train_raw = train_df.drop(columns="label").values
y_test_raw = test_df["label"].values
X_test_raw = test_df.drop(columns="label").values

# --- 5.2 Doublons exacts ---
def hash_images(X):
    # Une empreinte (hash) par image : deux images identiques ont la même empreinte
    return pd.util.hash_pandas_object(pd.DataFrame(X), index=False).values

h_train = hash_images(X_train_raw)
h_test = hash_images(X_test_raw)

dup_in_train = pd.Series(h_train).duplicated().values      # copie d'une image déjà vue dans le train
dup_in_test = pd.Series(h_test).duplicated().values
train_also_in_test = np.isin(h_train, h_test)               # image du train identique à une image du test

print("Doublons exacts dans le train         :", int(dup_in_train.sum()))
print("Doublons exacts dans le test          :", int(dup_in_test.sum()))
print("Images du train identiques au test    :", int(train_also_in_test.sum()))

keep = ~(dup_in_train | train_also_in_test)
X_train_clean = X_train_raw[keep]
y_train_clean = y_train_raw[keep]
print("Images du train retirées              :", int((~keep).sum()))
print("Images du train conservées            :", len(X_train_clean))

# --- 5.3 Réencodage des labels ---
original_labels = sorted(int(l) for l in np.unique(y_train_raw))
assert set(np.unique(y_test_raw)) <= set(original_labels), "Le test contient un label absent du train"
label_to_index = {l: i for i, l in enumerate(original_labels)}
class_names = [label_to_letter(l) for l in original_labels]
NUM_CLASSES = len(class_names)

print("\nCorrespondance label d'origine -> indice -> lettre :")
for l in original_labels:
    print(f"  {l:>2} -> {label_to_index[l]:>2} -> {label_to_letter(l)}")

y_train_all = np.array([label_to_index[int(l)] for l in y_train_clean])
y_test = np.array([label_to_index[int(l)] for l in y_test_raw])

# --- 5.4 et 5.5 Mise en forme 28x28x1 et normalisation ---
def preprocess_pixels(X):
    return (X.reshape(-1, 28, 28, 1) / 255.0).astype("float32")

X_train_all = preprocess_pixels(X_train_clean)
X_test = preprocess_pixels(X_test_raw)

# --- 5.6 Séparation train / validation stratifiée ---
X_train_full, X_val, y_train_full, y_val = train_test_split(
    X_train_all, y_train_all, test_size=0.2, stratify=y_train_all, random_state=SEED)

# --- 5.7 Sous-ensemble équilibré ---
def make_subset(X, y, n_per_class, seed=SEED):
    rng = np.random.default_rng(seed)
    idx = []
    for c in np.unique(y):
        cls_idx = np.where(y == c)[0]
        idx.extend(rng.choice(cls_idx, min(n_per_class, len(cls_idx)), replace=False))
    idx = rng.permutation(np.array(idx))
    return X[idx], y[idx]

X_train_300, y_train_300 = make_subset(X_train_full, y_train_full, 300)

# --- Vérifications ---
print("\nForme X_train_full :", X_train_full.shape, "| y :", y_train_full.shape)
print("Forme X_val        :", X_val.shape, "| y :", y_val.shape)
print("Forme X_test       :", X_test.shape, "| y :", y_test.shape)
print("Forme X_train_300  :", X_train_300.shape)
print("Type :", X_train_full.dtype, "| min =", X_train_full.min(), "| max =", X_train_full.max())
print("Indices de classes :", np.unique(y_train_full).min(), "à", np.unique(y_train_full).max(),
      "| nombre :", len(np.unique(y_train_full)))
print("Images par classe dans le sous-ensemble : min =", np.bincount(y_train_300).min(),
      "| max =", np.bincount(y_train_300).max())

# Contrôle visuel : la lettre affichée doit correspondre au signe
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, i in zip(axes, range(8)):
    ax.imshow(X_train_full[i, :, :, 0], cmap="gray", interpolation="nearest", vmin=0, vmax=1)
    ax.set_title(class_names[y_train_full[i]])
    ax.axis("off")
plt.suptitle("Images après prétraitement (contrôle des labels)")
plt.tight_layout()
plt.show()

# --- 5.8 Proximité validation / test par rapport au train ---
def nearest_train_distance(A, B):
    # Distance (RMS par pixel) entre chaque image de A et l'image la plus proche dans B
    A = A.reshape(len(A), -1); B = B.reshape(len(B), -1)
    d2 = (A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None, :] - 2 * A @ B.T
    return np.sqrt(np.maximum(d2.min(axis=1), 0) / A.shape[1])

rng = np.random.default_rng(SEED)
sample_val = X_val[rng.choice(len(X_val), min(500, len(X_val)), replace=False)]
sample_test = X_test[rng.choice(len(X_test), min(500, len(X_test)), replace=False)]
d_val = nearest_train_distance(sample_val, X_train_full)
d_test = nearest_train_distance(sample_test, X_train_full)

print("\nDistance à l'image du train la plus proche (500 images, pixels entre 0 et 1) :")
print(f"  Validation : médiane = {np.median(d_val):.4f} | moyenne = {d_val.mean():.4f}")
print(f"  Test       : médiane = {np.median(d_test):.4f} | moyenne = {d_test.mean():.4f}")

plt.figure(figsize=(7, 3.5))
bins = np.linspace(0, max(d_val.max(), d_test.max()), 40)
plt.hist(d_val, bins=bins, alpha=0.6, label="validation")
plt.hist(d_test, bins=bins, alpha=0.6, label="test")
plt.xlabel("Distance à l'image du train la plus proche")
plt.ylabel("Nombre d'images")
plt.title("Proximité avec le train : validation vs test")
plt.legend()
plt.tight_layout()
plt.savefig("results/proximite_val_test.png", dpi=200)
plt.show()


**Résultats obtenus :**

- `X_train_full` contient **21 964 images**, soit exactement 80 % de 27 455 : **aucune image n'a été retirée**. Il n'y a donc **ni doublon exact dans le train, ni image du train identique à une image du test**.
- Les formes sont correctes : images `28 × 28 × 1` en `float32` entre 0 et 1, labels de 0 à 23.
- **Proximité avec le train** (graphique `results/proximite_val_test.png`) : les images de **validation** sont presque toutes à une distance inférieure à 0,05 de leur plus proche voisine du train, alors que celles du **test** se situent surtout entre 0,05 et 0,15.

**Interprétation :** il n'y a pas de copie exacte, mais la validation contient des **variantes très proches** d'images du train (issues de la même photo d'origine). La validation sera donc **optimiste** ; le test, plus éloigné, est la seule mesure fiable de la capacité de généralisation.


## 6. Construction du CNN

**Responsable : Étudiante 1**

### Principe d'un CNN

- **Convolution (`Conv2D`)** : un **filtre** est une petite grille de poids (ici 3 × 3) qui glisse sur l'image. À chaque position, il multiplie les pixels par ses poids et fait la somme. Il produit ainsi une **carte de caractéristiques** (*feature map*) qui indique où se trouve un motif (bord, coin, courbe…). Les poids des filtres sont **appris** pendant l'entraînement.
- **Plusieurs filtres** par couche : 32 filtres donnent 32 cartes, chacune spécialisée dans un motif.
- **Activation ReLU** : `max(0, x)`. Elle ajoute de la **non-linéarité** ; sans elle, empiler des couches reviendrait à une seule opération linéaire.
- **Pooling (`MaxPooling2D`)** : garde la valeur maximale de chaque bloc 2 × 2. Les cartes deviennent deux fois plus petites, ce qui réduit les calculs et rend le réseau un peu tolérant aux petits décalages.
- **Empilement** : les premières couches détectent des motifs simples (bords), les suivantes les combinent en formes plus complexes (doigts, contours de la main). C'est pourquoi le nombre de filtres augmente (32 → 64 → 128).
- **`GlobalAveragePooling2D`** : fait la moyenne de chaque carte et produit un vecteur de 128 valeurs. On le préfère à `Flatten` car il crée beaucoup moins de paramètres, ce qui limite le surapprentissage.
- **`Dense` + `Dropout`** : une couche entièrement connectée combine les caractéristiques. Le **Dropout** désactive au hasard 30 % des neurones pendant l'entraînement pour éviter que le réseau mémorise les données.
- **Sortie `Dense(24, softmax)`** : un neurone par lettre. **Softmax** transforme les sorties en probabilités dont la somme vaut 1. La classe prédite est celle de probabilité maximale.

### Architecture et nombre de paramètres

`padding="same"` conserve la taille de l'image lors de la convolution.

| Couche | Sortie | Calcul des paramètres | Paramètres |
|---|---|---|---|
| Input | 28 × 28 × 1 | — | 0 |
| Conv2D 32, 3×3 | 28 × 28 × 32 | (3×3×1 + 1) × 32 | 320 |
| MaxPooling2D | 14 × 14 × 32 | — | 0 |
| Conv2D 64, 3×3 | 14 × 14 × 64 | (3×3×32 + 1) × 64 | 18 496 |
| MaxPooling2D | 7 × 7 × 64 | — | 0 |
| Conv2D 128, 3×3 | 7 × 7 × 128 | (3×3×64 + 1) × 128 | 73 856 |
| MaxPooling2D | 3 × 3 × 128 | — | 0 |
| GlobalAveragePooling2D | 128 | — | 0 |
| Dense 128, ReLU | 128 | (128 + 1) × 128 | 16 512 |
| Dropout 0,3 | 128 | — | 0 |
| Dense 24, Softmax | 24 | (128 + 1) × 24 | 3 096 |
| **Total** | | | **112 280** |

Le `+ 1` correspond au **biais** de chaque filtre ou neurone. Le total attendu doit être confirmé par `model.summary()`.


In [ ]:
from tensorflow.keras import layers

def build_model(num_classes=NUM_CLASSES):
    model = keras.Sequential([
        keras.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax"),
    ], name="asl_cnn")
    return model

model = build_model()
model.summary()
print("Nombre total de paramètres :", model.count_params())


**Résultat obtenu :** `model.summary()` confirme **112 280 paramètres entraînables**, conformément au calcul du tableau, et les dimensions attendues à chaque couche (28 → 14 → 7 → 3).


## 7. Compilation

**Responsable : Étudiante 2**

Compiler le modèle, c'est lui indiquer **comment apprendre** :

- **Optimiseur Adam** (taux d'apprentissage 0,001) : il ajuste les poids pour réduire l'erreur. Il adapte automatiquement le pas de chaque poids, ce qui le rend efficace sans réglage fin. 0,001 est sa valeur par défaut, un point de départ standard.
- **Fonction de perte `SparseCategoricalCrossentropy`** : elle mesure l'écart entre les probabilités prédites et la vraie classe. Plus la probabilité donnée à la bonne lettre est faible, plus la perte est grande. La version **Sparse** accepte des labels **entiers** (0 à 23), ce qui est notre cas ; la version sans « Sparse » demanderait un encodage *one-hot*.
- **Métrique accuracy** : la proportion d'images bien classées. Elle est facile à interpréter et adaptée ici car les classes sont à peu près équilibrées (section 4).


In [ ]:
def compile_model(model, learning_rate=1e-3):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )
    return model

model = compile_model(build_model())
print("Optimiseur :", type(model.optimizer).__name__, "| taux :", float(model.optimizer.learning_rate.numpy()))
print("Perte      :", type(model.loss).__name__)


## 8. Entraînement

**Responsable : Étudiante 2**

### Stratégie

On entraîne le même modèle avec **trois volumes de données** croissants :

| Expérience | Données d'entraînement |
|---|---|
| `300_par_classe` | 300 images par lettre (7 200 images) |
| `600_par_classe` | 600 images par lettre |
| `tout_le_train` | tout l'ensemble d'entraînement |

Pour chaque expérience, le modèle repart de zéro avec la même graine. On compare ensuite les résultats **sur la validation uniquement**. Le test n'est pas utilisé ici.

### Paramètres

- **Époques** : 40 au maximum. Une époque = un passage sur toutes les images d'entraînement.
- **Batch size** : 64. Les poids sont mis à jour après chaque paquet de 64 images, un compromis courant entre vitesse et stabilité.
- **EarlyStopping** : arrête l'entraînement si la perte de validation ne s'améliore plus pendant 5 époques, puis restaure les poids de la meilleure époque. Cela évite le surapprentissage et le temps perdu.
- **ModelCheckpoint** : enregistre le meilleur modèle (perte de validation la plus faible) de chaque expérience.

### Lecture des courbes

- **Surapprentissage** : la performance d'entraînement continue de s'améliorer alors que celle de validation stagne ou se dégrade.
- **Sous-apprentissage** : les deux restent mauvaises.


In [ ]:
import time

EPOCHS = 40
BATCH_SIZE = 64
os.makedirs("checkpoints", exist_ok=True)

experiments = {
    "300_par_classe": make_subset(X_train_full, y_train_full, 300),
    "600_par_classe": make_subset(X_train_full, y_train_full, 600),
    "tout_le_train": (X_train_full, y_train_full),
}

histories, summary_rows = {}, []
for name, (X_tr, y_tr) in experiments.items():
    print(f"\n===== Expérience : {name} ({len(X_tr)} images) =====")
    keras.utils.set_random_seed(SEED)
    exp_model = compile_model(build_model())
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(f"checkpoints/{name}.keras", monitor="val_loss", save_best_only=True),
    ]
    start = time.time()
    history = exp_model.fit(X_tr, y_tr, validation_data=(X_val, y_val),
                            epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks, verbose=2)
    duration = time.time() - start
    h = history.history
    best = int(np.argmin(h["val_loss"]))
    histories[name] = h
    summary_rows.append({
        "expérience": name,
        "images_train": len(X_tr),
        "époques_effectuées": len(h["loss"]),
        "meilleure_époque": best + 1,
        "train_accuracy": round(h["accuracy"][best], 4),
        "val_accuracy": round(h["val_accuracy"][best], 4),
        "train_loss": round(h["loss"][best], 4),
        "val_loss": round(h["val_loss"][best], 4),
        "durée_s": round(duration, 1),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("results/comparaison_experiences.csv", index=False)
print("\nComparaison des expériences (valeurs à la meilleure époque, sur la validation) :")
print(summary_df.to_string(index=False))

# --- Choix du modèle final : la plus petite perte de validation ---
best_name = summary_df.loc[summary_df["val_loss"].idxmin(), "expérience"]
print("\nModèle retenu (plus petite val_loss) :", best_name)
final_model = keras.models.load_model(f"checkpoints/{best_name}.keras")


### Courbes d'apprentissage


In [ ]:
fig, axes = plt.subplots(len(histories), 2, figsize=(12, 3.5 * len(histories)))
axes = np.atleast_2d(axes)
for row, (name, h) in zip(axes, histories.items()):
    ep = range(1, len(h["loss"]) + 1)
    row[0].plot(ep, h["accuracy"], label="train")
    row[0].plot(ep, h["val_accuracy"], label="validation")
    row[0].set_title(f"{name} — accuracy")
    row[0].set_xlabel("Époque"); row[0].legend()
    row[1].plot(ep, h["loss"], label="train")
    row[1].plot(ep, h["val_loss"], label="validation")
    row[1].set_title(f"{name} — loss")
    row[1].set_xlabel("Époque"); row[1].legend()
plt.tight_layout()
plt.savefig("results/courbes_apprentissage.png", dpi=200)
plt.show()


**Résultats obtenus (validation, à la meilleure époque) :**

| Expérience | Images d'entraînement | Époques effectuées | Meilleure époque | Accuracy train | Accuracy validation | Loss validation | Durée |
|---|---|---|---|---|---|---|---|
| 300 par classe | 7 200 | 40 | 35 | 0,9828 | 0,9985 | 0,0122 | 56 s |
| 600 par classe | 14 400 | 29 | 24 | 0,9922 | 0,9998 | 0,0016 | 41 s |
| Tout le train | 21 964 | 26 | 21 | 0,9984 | **1,0000** | **0,0003** | 56 s |

**Interprétation :**

- **Plus de données = apprentissage plus rapide et meilleur** : la meilleure époque arrive plus tôt (35 → 24 → 21) et la perte de validation diminue fortement. Avec 300 images par classe, l'entraînement atteint la limite de 40 époques sans que l'EarlyStopping se déclenche.
- **Pas de surapprentissage visible sur la validation** : les courbes train et validation restent proches et la validation ne se dégrade pas. L'accuracy de validation est même légèrement supérieure à celle du train, car le Dropout n'est actif que pendant l'entraînement.
- **Pas de sous-apprentissage** : toutes les expériences dépassent 98 % sur le train.
- **Limite :** la validation est **saturée** (99,85 à 100 %). Comme vu en section 5, elle contient des variantes très proches du train ; elle ne permet plus de départager finement les modèles ni d'estimer la performance réelle.
- **Modèle retenu :** `tout_le_train` (plus petite perte de validation).


## 9. Évaluation

**Responsable : Étudiante 2**

Le modèle retenu est évalué **une seule fois** sur le **test officiel** (7 172 images jamais utilisées pour l'entraînement ni pour les choix de réglage).

- **Accuracy** : proportion d'images bien classées.
- **Précision** d'une lettre : parmi les images prédites comme cette lettre, la part qui l'est vraiment.
- **Rappel** d'une lettre : parmi les vraies images de cette lettre, la part retrouvée par le modèle.
- **F1-score** : moyenne harmonique de la précision et du rappel ; il est faible dès que l'un des deux l'est.
- **Moyenne macro** : moyenne simple sur les 24 lettres (chaque lettre compte autant). **Moyenne pondérée** : chaque lettre pèse selon son nombre d'images.
- **Matrice de confusion** : ligne = vraie lettre, colonne = lettre prédite. La diagonale contient les bonnes réponses ; les cases hors diagonale montrent les confusions.


In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_fscore_support, accuracy_score)
import json

# --- Évaluation sur le test ---
test_loss, test_acc = final_model.evaluate(X_test, y_test, verbose=0)
y_proba = final_model.predict(X_test, verbose=0)
y_pred = y_proba.argmax(axis=1)

print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_acc:.4f}")

p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(y_test, y_pred, average="macro", zero_division=0)
p_w, r_w, f_w, _ = precision_recall_fscore_support(y_test, y_pred, average="weighted", zero_division=0)
print(f"Macro    : précision = {p_macro:.4f} | rappel = {r_macro:.4f} | F1 = {f_macro:.4f}")
print(f"Pondérée : précision = {p_w:.4f} | rappel = {r_w:.4f} | F1 = {f_w:.4f}")

# --- Rapport de classification ---
report = classification_report(y_test, y_pred, target_names=class_names, digits=4, zero_division=0)
print("\n" + report)
with open("results/rapport_classification.txt", "w") as f:
    f.write(report)

metrics = {"modele": str(best_name), "test_loss": float(test_loss), "test_accuracy": float(test_acc),
           "precision_macro": float(p_macro), "rappel_macro": float(r_macro), "f1_macro": float(f_macro),
           "precision_ponderee": float(p_w), "rappel_pondere": float(r_w), "f1_pondere": float(f_w)}
with open("results/metriques_test.json", "w") as f:
    json.dump(metrics, f, indent=2)

# --- Matrice de confusion ---
cm = confusion_matrix(y_test, y_pred, labels=range(NUM_CLASSES))
fig, ax = plt.subplots(figsize=(11, 10))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
ax.set_xlabel("Lettre prédite"); ax.set_ylabel("Vraie lettre")
ax.set_title("Matrice de confusion (test)")
plt.tight_layout()
plt.savefig("results/matrice_confusion.png", dpi=200)
plt.show()

# --- Lettres les mieux et les moins bien reconnues (F1) ---
p_c, r_c, f_c, s_c = precision_recall_fscore_support(y_test, y_pred, labels=range(NUM_CLASSES), zero_division=0)
per_class = pd.DataFrame({"lettre": class_names, "précision": p_c, "rappel": r_c, "f1": f_c, "images": s_c}).round(4)
per_class = per_class.sort_values("f1", ascending=False)
print("5 lettres les mieux reconnues :")
print(per_class.head(5).to_string(index=False))
print("\n5 lettres les moins bien reconnues :")
print(per_class.tail(5).to_string(index=False))


**Résultats obtenus sur le test (7 172 images) :**

| Métrique | Valeur |
|---|---|
| **Accuracy** | **96,17 %** |
| Loss | 0,1240 |
| Précision macro | 0,9574 |
| Rappel macro | 0,9646 |
| F1 macro | 0,9597 |
| F1 pondéré | 0,9617 |

**Interprétation :**

- Le modèle classe correctement **6 897 images sur 7 172** (275 erreurs).
- **Écart validation / test** : 100 % en validation contre 96,17 % en test. Cet écart est cohérent avec la section 5 : les images de test, issues de photos différentes, sont plus éloignées du train. **96,17 % est l'estimation honnête** de la performance sur des images de ce dataset.
- **Lettres parfaitement reconnues (F1 = 1,00)** : F, L et O ; quasi parfaites : P (0,999), C (0,998) et E (0,998).
- **Lettres les plus difficiles (F1)** : R (0,883), G (0,885), W (0,905), T (0,927), N (0,928) et M (0,934).
- Certaines lettres ont un **rappel de 99,5 à 100 % mais une précision plus faible** : A (0,94), I (0,91), Q (0,91), S (0,88) et W (0,83). Elles « attirent » des images d'autres lettres ; par exemple, W reçoit 41 images de B.


## 10. Analyse des erreurs

**Responsable : Étudiante 2**

On regarde **où** et **pourquoi** le modèle se trompe :

1. des exemples d'images **bien classées**, avec la confiance du modèle ;
2. des exemples d'images **mal classées** (vraie lettre / lettre prédite / confiance) ;
3. les **paires de lettres les plus confondues**, tirées de la matrice de confusion ;
4. pour les 3 paires principales, une image mal classée à côté d'une image correcte de la lettre prédite, pour comparer visuellement les deux signes.

Les explications visuelles des confusions (par exemple des doigts en position proche) sont des **hypothèses** issues de l'observation, pas des preuves.


In [ ]:
confidence = y_proba.max(axis=1)
correct_idx = np.where(y_pred == y_test)[0]
wrong_idx = np.where(y_pred != y_test)[0]
print(f"Bien classées : {len(correct_idx)} | Mal classées : {len(wrong_idx)} (sur {len(y_test)})")

def show_grid(indices, title, filename, n=16):
    indices = indices[:n]
    if len(indices) == 0:
        print("Aucune image à afficher pour :", title)
        return
    cols = 8
    rows = int(np.ceil(len(indices) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(14, 2.1 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, i in zip(axes, indices):
        ax.imshow(X_test[i, :, :, 0], cmap="gray", interpolation="nearest", vmin=0, vmax=1)
        color = "green" if y_pred[i] == y_test[i] else "red"
        ax.set_title(f"vrai {class_names[y_test[i]]} / prédit {class_names[y_pred[i]]}\n{confidence[i]:.0%}",
                     fontsize=8, color=color)
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(f"results/{filename}", dpi=200)
    plt.show()

rng = np.random.default_rng(SEED)
show_grid(rng.permutation(correct_idx), "Exemples d'images bien classées", "exemples_bien_classes.png", n=8)
show_grid(rng.permutation(wrong_idx), "Exemples d'images mal classées", "exemples_mal_classes.png", n=16)

# --- Paires les plus confondues ---
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)
pairs = [(cm_off[i, j], class_names[i], class_names[j], i, j)
         for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if cm_off[i, j] > 0]
pairs.sort(reverse=True)
confusions = pd.DataFrame([{"vraie lettre": a, "prédite": b, "nombre": n,
                            "% de la vraie lettre": round(100 * n / cm[i].sum(), 1)}
                           for n, a, b, i, j in pairs[:10]])
print("\n10 confusions les plus fréquentes :")
print(confusions.to_string(index=False) if len(confusions) else "aucune")

# --- Comparaison visuelle des 3 principales confusions ---
top3 = pairs[:3]
if top3:
    fig, axes = plt.subplots(len(top3), 2, figsize=(5, 2.6 * len(top3)))
    axes = np.atleast_2d(axes)
    for row, (n, a, b, i, j) in zip(axes, top3):
        err = np.where((y_test == i) & (y_pred == j))[0][0]
        ref_candidates = np.where((y_test == j) & (y_pred == j))[0]
        row[0].imshow(X_test[err, :, :, 0], cmap="gray", interpolation="nearest"); row[0].set_title(f"vrai {a}, prédit {b}", fontsize=9)
        if len(ref_candidates):
            row[1].imshow(X_test[ref_candidates[0], :, :, 0], cmap="gray", interpolation="nearest")
        row[1].set_title(f"exemple correct de {b}", fontsize=9)
        for ax in row:
            ax.axis("off")
    plt.tight_layout()
    plt.savefig("results/principales_confusions.png", dpi=200)
    plt.show()


**Résultats obtenus :** 6 897 images bien classées, 275 mal classées.

**Confusions les plus fréquentes (lues dans la matrice de confusion) :**

| Vraie lettre | Prédite | Nombre | % de la vraie lettre |
|---|---|---|---|
| B | W | 41 | 9,5 % |
| M | S | 31 | 7,9 % |
| Y | I | 27 | 8,1 % |
| K | R | 21 | 6,3 % |
| N | A | 21 | 7,2 % |
| H | G | 20 | 4,6 % |
| T | G | 20 | 8,1 % |
| G | H | 18 | 5,2 % |
| G | Q | 17 | 4,9 % |
| M | N | 14 | 3,6 % |

**Interprétation :**

- Les erreurs sont **concentrées** : ces 10 paires représentent **230 des 275 erreurs (84 %)**. Le modèle ne se trompe pas au hasard, mais sur des signes précis.
- *Hypothèses* tirées de l'observation des images (non démontrées) :
  - **M, N, S, A** sont des poings fermés qui diffèrent surtout par la **position du pouce**, un détail de quelques pixels à 28 × 28.
  - **G, H** (et **Q**) montrent l'index (et le majeur pour H) pointé sur le côté : la forme générale de la main est très proche.
  - **Y → I** : les deux lèvent l'auriculaire ; Y ajoute le pouce écarté, qui peut être peu visible.
  - **B → W** : plusieurs doigts levés dans les deux cas. Sur l'exemple affiché, une bande sombre verticale en arrière-plan pourrait aussi perturber le modèle.
- Ces confusions concernent des **signes réellement proches visuellement**, ce qui montre une limite de la résolution 28 × 28 en niveaux de gris.


## 11. Prédiction sur une nouvelle image

**Responsable : Étudiante 2**

Pour utiliser le modèle sur une image quelconque, il faut lui appliquer **exactement le même prétraitement** qu'aux données d'entraînement :

1. **Vérifier** que le fichier est bien une image lisible (JPEG, PNG…).
2. **Corriger l'orientation** des photos de téléphone (métadonnées EXIF).
3. **Convertir en niveaux de gris** (1 canal).
4. **Recadrer au centre** en carré, pour ne pas déformer la main.
5. **Redimensionner en 28 × 28**.
6. **Normaliser** en divisant par 255.
7. **Prédire** et afficher les probabilités des 24 lettres.

**Démonstration** : on enregistre une image du test au format PNG puis on la fait passer par toute la chaîne, comme une image externe.

**Avec votre propre photo** : mettez `UTILISER_MA_PHOTO = True` dans la deuxième cellule et exécutez-la. Pour de meilleurs résultats : main seule, cadrée au centre, fond uni, bonne lumière. Le modèle a été entraîné sur des images très particulières (recadrées, 28 × 28) : une photo prise dans d'autres conditions peut être mal classée.


In [ ]:
from PIL import Image, ImageOps

def preprocess_external_image(path):
    try:
        img = Image.open(path)
        img.verify()                     # vérifie que le fichier est une image valide
        img = Image.open(path)           # réouverture nécessaire après verify()
    except Exception as e:
        raise ValueError(f"Fichier non compatible : {e}")
    print(f"Image chargée : format = {img.format}, taille = {img.size}, mode = {img.mode}")
    img = ImageOps.exif_transpose(img)   # orientation des photos de téléphone
    img = img.convert("L")               # niveaux de gris
    w, h = img.size
    s = min(w, h)
    img = img.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s))  # carré centré
    img = img.resize((28, 28), Image.Resampling.LANCZOS)
    arr = np.asarray(img, dtype="float32") / 255.0
    return arr.reshape(1, 28, 28, 1)

def predict_image(path, model=None, top_k=5):
    model = model or final_model
    x = preprocess_external_image(path)
    proba = model.predict(x, verbose=0)[0]
    order = np.argsort(proba)[::-1]
    print(f"\nLettre prédite : {class_names[order[0]]} ({proba[order[0]]:.1%})")
    print(f"Top {top_k} :", ", ".join(f"{class_names[k]} {proba[k]:.1%}" for k in order[:top_k]))

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.2), gridspec_kw={"width_ratios": [1, 4]})
    axes[0].imshow(x[0, :, :, 0], cmap="gray", interpolation="nearest", vmin=0, vmax=1)
    axes[0].set_title(f"Prédit : {class_names[order[0]]}")
    axes[0].axis("off")
    colors = ["tab:green" if k == order[0] else "tab:blue" for k in range(len(class_names))]
    axes[1].bar(class_names, proba, color=colors)
    axes[1].set_ylim(0, 1)
    axes[1].set_ylabel("Probabilité")
    axes[1].set_title("Scores des 24 classes")
    plt.tight_layout()
    plt.show()
    return class_names[order[0]], proba

# --- Démonstration avec une image du test enregistrée en PNG ---
demo_index = 0
demo_path = "results/exemple_image_test.png"
Image.fromarray((X_test[demo_index, :, :, 0] * 255).astype("uint8")).save(demo_path)
print("Vraie lettre de l'image de démonstration :", class_names[y_test[demo_index]])
predicted_letter, scores = predict_image(demo_path)


In [ ]:
# --- Tester avec votre propre photo ---
UTILISER_MA_PHOTO = False   # mettre True puis exécuter cette cellule

if UTILISER_MA_PHOTO:
    from google.colab import files
    uploaded = files.upload()          # une fenêtre permet de choisir le fichier
    for filename in uploaded:
        predict_image(filename)
else:
    print("Pour tester votre photo : UTILISER_MA_PHOTO = True, puis exécuter la cellule.")


**Résultat obtenu :** l'image de démonstration, enregistrée en PNG puis repassée par toute la chaîne (vérification, niveaux de gris, recadrage, 28 × 28, normalisation), est prédite **G avec une probabilité de 97,2 %**. La chaîne de prédiction sur fichier externe fonctionne.

Aucune photo personnelle n'a été testée dans cette exécution. Une photo prise dans d'autres conditions (cadrage, fond, lumière) serait probablement moins bien reconnue : c'est une limite discutée en conclusion.


## 12. Sauvegarde et réutilisation

**Responsable : Étudiante 2**

- Le modèle est enregistré au format **`.keras`**, le format natif de Keras 3. Il contient l'architecture, les poids et la configuration de compilation.
- Le fichier **`classes.json`** conserve l'**ordre des classes** (indice → lettre), la correspondance avec les labels d'origine et les **paramètres de prétraitement**. Sans lui, une sortie « 9 » du réseau ne pourrait pas être traduite en lettre (ici K, et non J).
- On **recharge** ensuite le modèle et on vérifie qu'il donne exactement les mêmes prédictions.
- Enfin, les dossiers `model/` et `results/` sont compressés en `resultats_asl.zip` et téléchargés, pour être ajoutés au dépôt GitHub.


In [ ]:
os.makedirs("model", exist_ok=True)

# --- Sauvegarde du modèle ---
model_path = "model/asl_cnn.keras"
final_model.save(model_path)

# --- Sauvegarde des classes et du prétraitement ---
classes_info = {
    "class_names": class_names,
    "index_to_original_label": {i: l for l, i in label_to_index.items()},
    "preprocessing": {
        "input_shape": [28, 28, 1],
        "color": "grayscale",
        "resize": "recadrage carré centré puis 28x28",
        "normalization": "pixel / 255",
    },
    "modele_retenu": str(best_name),
}
with open("model/classes.json", "w") as f:
    json.dump(classes_info, f, indent=2, ensure_ascii=False)

print("Fichiers enregistrés :")
for p in ["model/asl_cnn.keras", "model/classes.json"]:
    print(f"  {p} : {os.path.getsize(p) / 1024:.1f} Ko")

# --- Rechargement et vérification ---
reloaded_model = keras.models.load_model(model_path)
with open("model/classes.json") as f:
    reloaded_classes = json.load(f)["class_names"]

same = np.allclose(final_model.predict(X_test[:200], verbose=0),
                   reloaded_model.predict(X_test[:200], verbose=0), atol=1e-6)
print("Prédictions identiques après rechargement :", same)
print("Ordre des classes identique :", reloaded_classes == class_names)


In [ ]:
# --- Téléchargement des résultats (à ajouter ensuite sur GitHub) ---
import shutil
shutil.make_archive("resultats_asl", "zip", ".", "model")
shutil.make_archive("resultats_results", "zip", ".", "results")
try:
    from google.colab import files
    files.download("resultats_asl.zip")
    files.download("resultats_results.zip")
except ImportError:
    print("Hors Colab : archives créées dans le dossier courant.")


**Résultats obtenus :**

- `model/asl_cnn.keras` : 1 362 Ko ; `model/classes.json` : 0,8 Ko.
- Après rechargement, les **prédictions sont identiques** et l'**ordre des classes est conservé**.
- Le modèle peut donc être réutilisé sans réentraînement.


## 13. Conclusion

**Responsable : Commun**

### Réponse à la problématique

Oui : un CNN simple (3 couches de convolution, **112 280 paramètres**) reconnaît les 24 lettres statiques de l'ASL de ce dataset avec une **accuracy de 96,17 %** et un **F1 macro de 0,960** sur 7 172 images de test indépendantes, après moins d'une minute d'entraînement sur GPU.

### Principaux enseignements

1. **Vérifier les données avant tout** : les labels n'étaient pas consécutifs (le 9 manque), ce qui imposait un réencodage. Aucune valeur manquante ni doublon exact n'a été trouvé.
2. **Plus de données améliore l'apprentissage** : convergence plus rapide et perte de validation plus faible en passant de 300 images par classe à tout le train.
3. **La validation peut être trompeuse** : elle atteint 100 % alors que le test donne 96,17 %. La mesure de proximité a montré que la validation contient des variantes très proches du train. Garder le test **intact et utilisé une seule fois** était donc indispensable.
4. **Les erreurs sont structurées** : 84 % d'entre elles concernent 10 paires de lettres visuellement proches (M/N/S/A, G/H/Q, T/G, Y/I, B/W, K/R).

### Limites

- **Dataset restreint** : 34 627 images, mais issues de seulement **1 704 photos** ; peu de diversité de personnes, de fonds et d'éclairages.
- **Images très normalisées** (main recadrée, 28 × 28, niveaux de gris) : les résultats **ne prouvent pas** une reconnaissance fiable de l'ASL sur des photos ou une webcam en conditions réelles.
- **Validation optimiste** : sans identifiant de la photo d'origine, il est impossible de séparer train et validation par photo.
- **Périmètre** : J et Z (lettres en mouvement), mots, phrases, expressions du visage et grammaire de l'ASL ne sont pas couverts. Le système **ne traduit pas** la langue des signes.
- Sur GPU, une nouvelle exécution peut donner des chiffres très légèrement différents. Les valeurs ci-dessus sont celles de l'exécution du 8 novembre 2026.

### Perspectives

- **Augmentation de données** pendant l'entraînement (rotations, décalages, zoom, luminosité) pour mieux généraliser.
- **Test sur des photos réelles** ou un autre dataset ASL pour mesurer la robustesse hors de ce dataset.
- **Résolution plus élevée** pour distinguer les détails fins (position du pouce).
- **Explicabilité** (par exemple Grad-CAM) pour vérifier que le modèle regarde bien la main et non l'arrière-plan.
